# **Notebook 6: Interpretable Machine Learning**

**Winning model:** `RF_tuned` (selected by NB5). The analysis retains the Random Forest baseline-versus-tuned comparison. No single-respondent CP plots or model reselection.

**ICE visualization sections:** (1) regular ICE — pooled and by actual MNDS/HDS; (2) centered ICE — pooled and by actual MNDS/HDS (numeric predictors). Thin vertical **rug marks** along the bottom of the axes show observed predictor values among the held-out **test respondents represented by that panel**; their concentration indicates where more values are observed. For coded categories, the marks are jittered *horizontally for display only*, and category counts are also available.

### **Data and selected model**

In [ ]:
from pathlib import Path
import json, re, textwrap, math
import numpy as np
import pandas as pd
import joblib
import hashlib, platform, sys, sklearn, matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, recall_score, roc_auc_score,
                             balanced_accuracy_score)
from IPython.display import display

PROJECT_ROOT = Path.cwd()
OUT_DIR = PROJECT_ROOT / "NB_OUTS1_TRAINONLY"

assert (OUT_DIR / "NB5_model_selection.json").exists(), \
    "NB5 outputs not found. Run NB5 first."

selection=json.loads((OUT_DIR/'NB5_model_selection.json').read_text())
SELECTED_MODEL=selection['selected_model']
FAMILY=SELECTED_MODEL.split('_')[0]
TRAIN_FILE={'LR':'NB3_LR_lasso_train.csv','RF':'NB3_RandomForest_train.csv','ANN':'NB3_ANN_train.csv'}[FAMILY]
TEST_FILE={'LR':'NB3_LR_lasso_test.csv','RF':'NB3_RandomForest_test.csv','ANN':'NB3_ANN_test.csv'}[FAMILY]

train_df=pd.read_csv(OUT_DIR/TRAIN_FILE,index_col='orig_index')
test_df=pd.read_csv(OUT_DIR/TEST_FILE,index_col='orig_index')

FEATURES=[c for c in train_df.columns if c!='depressed_flag']
assert FEATURES==[c for c in test_df.columns if c!='depressed_flag']

X_train=train_df[FEATURES].astype(float).copy()
X_test=test_df[FEATURES].astype(float).copy()
y_train=train_df['depressed_flag'].astype(int)
y_test=test_df['depressed_flag'].astype(int)

model=joblib.load(OUT_DIR/f'NB4_fitted_{SELECTED_MODEL}.joblib')
ranking=pd.read_csv(OUT_DIR/'NB5_training_only_ranking.csv').set_index('model')
THRESHOLD=float(ranking.loc[SELECTED_MODEL,'threshold'])
proba=model.predict_proba(X_test)[:,1]
pred=(proba>=THRESHOLD).astype(int)
assert len(X_test)==133 and len(X_train)==530

# Cross-check against NB5's saved test predictions.
nb5=pd.read_csv(OUT_DIR/'NB5_combined_predictions.csv')
nb5_test=nb5[nb5['split'].eq('test')].set_index('original_idx').loc[X_test.index]
assert np.allclose(proba,nb5_test[SELECTED_MODEL+'_proba'].to_numpy(),atol=1e-8)
assert np.array_equal(pred,nb5_test[SELECTED_MODEL+'_pred'].to_numpy())

NB6_DIR=OUT_DIR/'NB6_interpretation'
NB6_DIR.mkdir(exist_ok=True,parents=True)
print('Selected model:',SELECTED_MODEL,'| Features:',len(FEATURES),'| Threshold:',THRESHOLD)
print('Confusion matrix:\n',confusion_matrix(y_test,pred))
print('Test recall:',round(recall_score(y_test,pred),4),
      '| AUC:',round(roc_auc_score(y_test,proba),4),
      '| balanced accuracy:',round(balanced_accuracy_score(y_test,pred),4))

### **Winning model**

**RF_tuned is the winning model**, selected by NB5 using training cross-validation criteria. NB6 reads NB5's saved model selection, threshold, fitted estimator, and held-out test predictions; it does **not** reselect or retrain the model. The code above prints the current number of RF predictors and performance metrics (rather than retaining the obsolete hard-coded predictor counts and threshold). Group colors below refer to respondents' **observed CES-D-derived HDS/MNDS status**, not their model predictions. HDS is a symptom classification, **not** a clinical diagnosis.

### **Predictor grids and display settings**

In [ ]:
WINNING_MODEL = 'RF_tuned'
if SELECTED_MODEL != WINNING_MODEL:
    raise ValueError(f'NB5 currently selects {SELECTED_MODEL}, not {WINNING_MODEL}. '
                     'Check saved NB5 artifacts; do not force model selection in NB6.')

TOP_N = None            
GRID_POINTS = 50
SENTINEL = -1.0
SHOW_INDIVIDUAL_PLOTS = True 
SHOW_RUG_MARKS = True       
RUG_SEED = 42            
RUG_HEIGHT = (0.015, 0.060) 
COMBINED_PER_FIGURE = 12
ALL_IN_ONE_COLUMNS = 4
NB6_SEED = 42

ORDINAL_FEATURES = set(['C1A', 'C1B', 'C1C', 'C1D', 'C1E', 'C1F', 'C1G', 'C1H', 'C1I', 'C1J', 'C2A', 'C2B', 'C2C', 'C2D', 'C2E', 'C2F', 'C2G', 'C2H', 'D8A', 'D8B', 'D8C', 'D8D', 'D8E', 'D8F', 'D8G', 'D8H', 'D8I', 'B31A', 'B31F', 'G94', 'G95', 'G1', 'G2', 'C3', 'C4', 'C5', 'B2', 'B15', 'B16', 'B17', 'D9', 'G11', 'A10', 'G18A', 'G18D', 'G18K', 'G18O', 'G17A', 'A3_grp', 'windex'])
NOMINAL_FEATURES = set(['D1', 'D6', 'D13_PSCED', 'D37_PSOC', 'A5', 'A8', 'B37A', 'B37B', 'B37C', 'B37E', 'G32', 'L12', 'marstat', 'sogie', 'REG'])
BINARY_FEATURES = set(['A1', 'URBAN', 'urbanex', 'B4_1', 'B4_2', 'A9', 'A13', 'A18', 'A19', 'A25', 'A26A', 'A26E', 'A26F', 'A33', 'D10', 'D21', 'D27', 'D29', 'D30', 'D46', 'D51', 'D32', 'E4', 'E13', 'E14', 'E1_01', 'E11Q', 'F1', 'F5', 'F7', 'F11', 'G9', 'G21', 'G24', 'G27', 'G33', 'G36', 'G44', 'G48', 'G60', 'G61', 'G68', 'G72', 'G87', 'G88', 'G101', 'G102', 'I113', 'I114', 'I133C', 'I140', 'I105', 'I17', 'J2', 'J4', 'abs_abortion', 'vaxstat', 'ever_sex', 'N12D', 'B19C', 'B19F', 'B19G', 'G56B'])
CONTINUOUS_FEATURES = set(['se_score', 'sf_score', 'windex_score', 'bmi_score', 'A3', 'G14', 'G16', 'I42', 'L5', 'G28', 'J55', 'age_first_sex', 'E7'])
COUNT_FEATURES = set(['B1', 'F2A', 'F2B', 'I102', 'A11'])
NUMERIC_SENTINEL_FEATURES = set(['I102', 'I42', 'L5', 'G28', 'J55', 'B19F', 'B19G', 'B19C', 'F2A', 'F2B', 'G56B', 'E7', 'E10E', 'E10B', 'A11', 'N12D', 'A26F', 'A26A', 'A26E'])
NUMERIC_ORDINAL_FEATURES = {'C3','C4','C5'}
DATA_DICTIONARY_LABELS = {'CASEID': 'Unique individual identifier', 'G94': 'G94. Ever thought of suicide', 'se_score': 'Self-esteem score', 'G95': 'G95. Ever tried to commit suicide', 'D1': 'D1. Enrolled in AY 2020-2021', 'B31A': 'a. Family members are supportive of each other', 'G1': 'G1. State of health', 'C5': 'C5. Happiness', 'B31F': 'f. Family members eat together at least one meal a day', 'G24': 'G24. Ever tried smoking', 'I113': 'I113. Any sexual contact with the same sex', 'G61': 'G61. Ever got in trouble due to drinking', 'sf_score': 'Self-efficacy score', 'C3': 'C3. Satisfaction with life as a whole', 'I102': 'I102. No. of sexual partners (last 12 mos.)', 'B4_1': 'B4. Father is still alive', 'E13': 'E13. Ever been harassed using technology', 'K26': 'K26. About the right age for pregnancy', 'A13': 'A13. Take part in religious activities', 'B2': 'B2. Gets along well with siblings', 'I133C': '', 'D13_PSCED': 'D13. College course/graduate degree completed', 'D10': 'D10. Ever received an award in school', 'D6': 'D6. Type of school', 'I42': 'I42. Age at first sex', 'D37_PSOC': 'D37. (EDITED) Current work', 'L5': 'L5. Age first realized sexual orientation', 'A9': 'A9. Attend religious services', 'G28': 'G28. Age started smoking regularly', 'J55': 'J55. Age had first child', 'D46': 'D46. Participated in community activity', 'C4': 'C4. Satisfaction with financial situation of family', 'G87': 'G87. Anyone hurt you physically', 'D3': 'D3. Level of schooling', 'E14': 'E14. Any of those harassment sexual in nature', 'D32': 'D32. Looked for work or try business in the past week', 'B37C': 'B37c. Problem regarding intimate relationships', 'A18': 'A18. Pray together at home', 'B14': 'B14. Parents living together', 'G16': 'G16. No. of hours of sleep', 'I105': 'I105. Ever tried masturbating', 'G32': 'G32. Smoking behavior', 'G2': 'G2. State of health now', 'G18A': 'G18. FOOD AND DRINKS (Fresh fruit)', 'I140': 'I140. Ever received sex photos/videos', 'G17A': 'G17. Regular meals (Breakfast)', 'A8': 'A8. Religion', 'B8A_2': "B8a. Mother's highest grade completed", 'B8A_1': "B8a. Father's highest grade completed", 'C1A': '', 'C1B': 'C1b. You feel that you have a number of good qualities.', 'C1C': 'C1c. All in all, you are inclined to think that you are a failure.', 'C1D': 'C1d. You are able to do things as well as most other people.', 'C1E': 'C1e. You feel you do not have much to be proud of.', 'C1F': 'C1f. You take a positive attitude toward yourself.', 'C1G': 'C1g. On the whole, you are satisfied with yourself.', 'C1H': 'C1h. You wish you could have more respect for yourself.', 'C1I': 'C1i. You certainly feel useless at times.', 'C1J': 'C1j. At times you think you are no good at all.', 'C2A': 'C2a. You will be able to achieve most of the goals that you set for yourself.', 'C2B': 'C2b. When facing difficult tasks, you are certain that you will accomplish them.', 'C2C': 'C2c. In general, you think that you can obtain outcomes that are important to you.', 'C2D': 'C2d. You believe you can succeed at most any endeavor to which you set your mind.', 'C2E': 'C2e. You will be able to successfully overcome many challenges.', 'C2F': 'C2f. You are confident that you can perform effectively on many different tasks.', 'C2G': 'C2g. Compared to other people, you can do most tasks very well.', 'C2H': 'C2h. Even when things are tough, you can perform quite well.', 'D8A': 'D8a. Problems with access to devices when you needed them', 'D8B': 'D8b. Problems with internet access', 'D8C': 'D8c. Problems with access to school supplies (e.g. paper, pencil)', 'D8D': 'D8d. Problems with access to modules and other learning materials', 'D8E': 'D8e. Problems with finding a quiet place to study', 'D8F': 'D8f. Problems with finding time to study because of household responsibilities', 'D8G': 'D8g. Problems with motivating yourself to do school work', 'D8H': 'D8h. Problems with understanding your school lessons', 'D8I': 'D8i. Problems with finding someone who could help you with your school work', 'L12': 'L12. Gender identity', 'A1': 'A1. Sex', 'A3': 'A3. Age (in years)', 'A3_grp': 'Age (grouped)', 'hgc1': 'Highest grade completed (grouped)', 'hgc3': 'Highest grade completed (7 categories)', 'marstat': 'Marital status (grouped)', 'URBAN': 'Urbanity', 'urbanex': 'If with urban exposure', 'windex': 'Wealth index', 'windex_score': 'Wealth index factor score (5 decimals)', 'sogie': 'Sexual and gender minorities with gender nonconforming appearance', 'REG': 'Region', 'A5': 'A5. Main activity', 'B1': 'B1. Total no. of siblings', 'B4_2': 'B4. Mother is still alive', 'B15': 'B15. Parents getting along', 'B16': 'B16. Getting along with father', 'B17': 'B17. Getting along with mother', 'B19F': 'B19. Reason raised by [B18] - Dad works outside the country', 'B19G': 'B19. Reason raised by [B18] - Mom works outside the country', 'B19C': 'B19. Reason raised by [B18] - Parents separated/divorced', 'B37A': 'B37a. Problem regarding relationship with family members', 'B37E': 'B37e. Financial problems', 'F1': 'F1. Close friends before pandemic', 'F2A': 'F2a. No. of male close friends', 'F2B': 'F2b. No. of female close friends', 'F5': 'F5. Share problems with close friends', 'F7': 'F7. Online friends', 'F11': 'F11. Share problems on social media', 'B37B': 'B37b. Problem regarding relationship with friends', 'D9': 'D9. Performance in school', 'D21': 'D21. Ever been suspended', 'D27': 'D27. Ever repeated a grade/level from school', 'D29': 'D29. Ever worked for pay', 'D30': 'D30. Worked during the past week', 'D51': 'D51. Volunteer work', 'bmi_score': 'BMI score', 'bmi_cat': 'BMI category', 'G5': 'G5. Self in terms of weight', 'G7': 'G7. Weight, want to be', 'G9': 'G9. Physical exercise', 'G11': 'G11. Frequency of exercise', 'G14': 'G14. Time to fall asleep (minutes)', 'G18D': '', 'G18K': 'G18. FOOD AND DRINKS (Fast food (e.g., hamburgers, pizza))', 'G18O': 'G18. FOOD AND DRINKS (Soda (soft drinks) and sweetened drinks)', 'G21': 'G21. Traveled for leisure', 'G27': 'G27. Currently smoke', 'G33': 'G33. Ever used e-cigarette', 'G36': 'G36. Currently vape', 'G44': 'G44. Ever tried drinking alcohol', 'G48': 'G48. Currently drink', 'G60': 'G60. Ever experience e-numan', 'G68': 'G68. Ever tried drugs', 'G72': 'G72. Currently using drugs', 'G56B': 'G56. Reason for drinking - To escape from problems/concerns', 'G101': 'G101. Suicide in the family', 'G102': 'G102. Know any suicide prevention services', 'G88': 'G88. Hurt you physically (last 12 mo)', 'ever_sex': '[COMPUTED] Ever had sex', 'age_first_sex': '[COMPUTED] Age of first sex', 'I114': 'I114. Ever been forced to have sex', 'J2': 'J2. Ever been pregnant', 'J4': 'J4. Currently pregnant', 'I17': 'I17. Ever had heterosexual relationship', 'K6': 'K6. Divorce', 'abs_abortion': 'Absolute disapproval of abortion by R (Does not approve in all circumstances)', 'E4': 'E4. Internet use', 'E7': 'E7. Duration of internet use in a week (in hours)', 'E10E': 'e. Tiktok', 'E10B': 'b. Facebook', 'E1_01': 'E1. Personally own gadget - Smartphone', 'E11Q': 'q. Interact/chat with strangers or random people online', 'A10': 'A10. Frequency - religious services', 'A11': 'A11. Frequency - prayer in a day', 'A19': 'A19. Member of religious organization', 'vaxstat': '', 'N13': 'N13. Vaccinated', 'N18B': 'b. Avoiding non-essential social gatherings', 'N12D': 'N12. Experiences - Talked badly or gossiped about', 'A26F': 'A26. Reason moved - COVID-19 pandemic', 'A25': 'A25. Ever moved since 2016', 'A26A': 'A26. Reason moved - Typhoon', 'A26E': 'A26. Reason moved - Peace and order problem', 'A33': 'A33. Ever lived away from family'}

level_labels = {}
level_candidates = [
    OUT_DIR/'NB1_RF_ANN_level_order.csv',
    OUT_DIR.parent/'NB_OUTS1'/'NB1_RF_ANN_level_order.csv',
    Path('/home/kathleen/THESIS1/NB_OUTS1/NB1_RF_ANN_level_order.csv'),
]
LEVEL_FILE = next((p for p in level_candidates if p.exists()), None)
if LEVEL_FILE is None:
    raise FileNotFoundError('NB1_RF_ANN_level_order.csv not found; rerun NB1/NB3 or restore its original category mapping.')
labels = pd.read_csv(LEVEL_FILE, keep_default_na=False)
level_labels = {(str(r.feature),float(r.code)):str(r.level) for r in labels.itertuples()}
CATEGORICAL_FEATURES = {feat for feat,_ in level_labels}
print('Loaded NB1 category lookup:', LEVEL_FILE)

if FAMILY == 'RF':
    fi = pd.DataFrame({'feature':FEATURES,'importance':model.feature_importances_}).sort_values('importance',ascending=False)
elif FAMILY == 'LR':
    fi = pd.DataFrame({'feature':FEATURES,'importance':np.abs(model.named_steps['lr'].coef_[0])}).sort_values('importance',ascending=False)
else:
    fi = pd.DataFrame({'feature':FEATURES,'importance':np.nan})
if FAMILY != 'ANN':
    fi['importance_share'] = fi['importance']/fi['importance'].sum()
    fi['cumulative_importance'] = fi['importance_share'].cumsum()
print('RF feature ranking (fitted winner):')
print(fi.head(25).round(4).to_string(index=False))
plot_features = fi['feature'].tolist() if TOP_N is None else fi['feature'].head(TOP_N).tolist()
print(f'Plotting {len(plot_features)} of {len(FEATURES)} selected features')

NUMERIC_KINDS = {'continuous','count','numeric_discrete'}
ORDINAL_KINDS = {'ordinal'}
CATEGORICAL_KINDS = {'ordinal','nominal','binary','categorical_review','missing_only'}

def choose_grid_kind(feat, values):
    values = np.asarray(values,dtype=float)
    declared = ('ordinal' if feat in ORDINAL_FEATURES else
                'nominal' if feat in NOMINAL_FEATURES else
                'binary' if feat in BINARY_FEATURES else
                'continuous' if feat in CONTINUOUS_FEATURES else
                'count' if feat in COUNT_FEATURES else None)
    # Only known actual missing encodings (from NB1) are interpreted as sentinel.
    sentinel_is_missing = bool(feat in CATEGORICAL_FEATURES or
                               feat in NUMERIC_SENTINEL_FEATURES)
    has_sentinel = sentinel_is_missing and bool(np.any(np.isclose(values,SENTINEL)))
    valid = np.sort(np.unique(values[~np.isclose(values,SENTINEL)] if sentinel_is_missing else values))
    if not len(valid):
        return 'missing_only',np.array([SENTINEL]),has_sentinel,'No nonmissing levels'
    # For binary/categorical/ordinal: enumerate actual train-level values, never interpolate.
    if feat in NUMERIC_ORDINAL_FEATURES:
        kind='ordinal'
    elif declared=='ordinal':
        kind='ordinal' if len(valid)>2 else 'binary'
    elif declared in ('nominal','binary'):
        kind='binary' if len(valid)==2 else ('categorical_review' if declared=='binary' else 'nominal')
    elif declared=='count':
        kind='count'
    elif declared=='continuous':
        kind='continuous' if len(valid)>12 else 'numeric_discrete'
    elif feat in CATEGORICAL_FEATURES:
        kind='binary' if len(valid)==2 else 'categorical_review'
    else:
        # Numeric code without known semantic units: do not invent a continuous axis.
        kind='numeric_discrete' if len(valid)>12 else 'categorical_review'
    if kind in NUMERIC_KINDS:
        if kind=='continuous' and len(valid)>12:
            grid=np.linspace(valid.min(),valid.max(),GRID_POINTS)
        else:
            grid=valid
        # A missing sentinel is NOT an adjacent numerical value and is omitted from numeric sweeps.
    else:
        grid=np.r_[[SENTINEL],valid] if has_sentinel else valid
    return kind,np.asarray(grid,dtype=float),has_sentinel, ('Variable definition; check NB1 response labels' if declared or feat in NUMERIC_ORDINAL_FEATURES else 'Requires response-level verification')

feature_kind = {};grids={};review_rows=[]
for feat in FEATURES:  # AUDIT EVERY selected RF predictor, even if TOP_N limits figures
    vals=X_train[feat].dropna().to_numpy(dtype=float)
    kind,grid,has_sentinel,reason=choose_grid_kind(feat,vals)
    feature_kind[feat]=kind;grids[feat]=grid
    mapped=feat in CATEGORICAL_FEATURES
    review_rows.append({
        'feature':feat,'description':DATA_DICTIONARY_LABELS.get(feat,''),
        'ice_kind':kind,'review_basis':reason,'NB1_label_encoded':mapped,
        'n_valid_train_values':len(np.unique(vals[~np.isclose(vals,SENTINEL)])) if
            (mapped or feat in NUMERIC_SENTINEL_FEATURES) else len(np.unique(vals)),
        'sentinel_in_train':has_sentinel,
        'missing_sentinel_in_grid':bool(has_sentinel and kind in CATEGORICAL_KINDS),
        'levels': ' | '.join(f'{v:g} = {level_labels.get((feat,float(v)),"Missing/skip/unseen" if v==SENTINEL else "(numeric code)")}' for v in grid),
        'requires_response_label_review':kind in {'categorical_review','nominal'} or
            (kind=='ordinal' and not mapped and feat not in NUMERIC_ORDINAL_FEATURES)
    })
review_df=pd.DataFrame(review_rows)
review_df.to_csv(NB6_DIR/'NB6_predictor_type_audit.csv',index=False)
print('ICE predictor types (all selected RF predictors):')
print(review_df['ice_kind'].value_counts().to_string())
print('Categories needing response-level review:')
print(review_df.loc[review_df.requires_response_label_review,['feature','description','ice_kind']].to_string(index=False))
print('Saved:',NB6_DIR/'NB6_predictor_type_audit.csv')

def level_label(feat,value):
    if np.isclose(value,SENTINEL) and (feat in CATEGORICAL_FEATURES or feat in NUMERIC_SENTINEL_FEATURES):
        return 'Missing / skip / unseen'
    lab=level_labels.get((feat,float(value)))
    return lab if lab is not None else (str(int(value)) if float(value).is_integer() else f'{value:.3g}')

def xvals(feat):
    return grids[feat] if feature_kind[feat] in NUMERIC_KINDS else np.arange(len(grids[feat]))

def format_axis(ax,feat,actual_group=None):
    if feature_kind[feat] in NUMERIC_KINDS: return
    text=[]
    for value in grids[feat]:
        label=level_label(feat,value)
        if actual_group is not None:
            mask=(y_test.to_numpy()==int(actual_group))
            n=int(np.isclose(X_test.loc[mask,feat].to_numpy(dtype=float),value).sum())
            label+=f'\n(n={n})'
        text.append(textwrap.fill(label,18))
    ax.set_xticks(np.arange(len(text)))
    ax.set_xticklabels(text,rotation=30 if any(len(t)>13 for t in text) else 0,
                       ha='right' if any(len(t)>13 for t in text) else 'center',fontsize=8)

def sweep(base_rows,feat):
    # Pure ICE: change one predictor at a time; no CP case selection/retraining.
    g=grids[feat];n=len(base_rows);k=len(g)
    changed=pd.DataFrame(np.repeat(base_rows[FEATURES].to_numpy(),k,axis=0),columns=FEATURES)
    changed[feat]=np.tile(g,n)
    return model.predict_proba(changed)[:,1].reshape(n,k)

def safe_name(s):return re.sub(r'[^A-Za-z0-9_-]','_',s)


### **Predictor-type and response-label audit**

All selected RF predictors are audited using YAFS5 variable descriptions and **NB1's original encoded-label lookup**, saved as `NB6_predictor_type_audit.csv`. Ordered categories (including `C1A`–`C1J`, `C2A`–`C2H`, `D8A`–`D8I`, `C3`–`C5`) use observed ordered response levels rather than an interpolated continuous axis. Nominal categories use separate dots; a numerical code never proves order. Some response-level meanings (e.g., school qualifications, reason categories, perception of weight) must still be verified against the full YAFS5 response codebook. The dictionary provided here contains variable *descriptions*, not every category code/label.

For numeric variables with known NB1 `-1` missing sentinels, the sentinel is excluded from numeric sweep axes; category-level missing/skip/unseen status is displayed as a discrete category when present. A generated ICE grid may still create unrealistic feature combinations when predictors are dependent; interpret patterns as **model behavior, not causal relationships** (Molnar, ICE limitations).

### **1. Regular ICE — absolute predicted probability**

For each respondent in the held-out test set, vary **one predictor** over a training-derived grid and hold every other predictor constant. Blue represents **actual MNDS** and orange represents **actual HDS** (not predicted classes). The dashed line is the NB5 decision threshold. Rug ticks inside the bottom of each frame show observed **test-set** predictor values; only the colors and observations change between pooled and split panels.

### **Standard and centered ICE**

Standard ICE draws each test respondent's predicted probability across a **training-derived** grid of one predictor while holding the other predictors fixed (Molnar, Chapter 13). A dashed line denotes the winning RF's **NB5 decision threshold**. Centered ICE (c-ICE) subtracts the prediction at the first valid numeric grid value and is shown only for numeric predictors; c-ICE differences are not absolute HDS probabilities. For nominal predictors, Molnar's discrete category treatment motivates dots rather than misleading connected lines; ordered scales may have lines only across *valid ordered levels*, never across a missing sentinel. Colors group respondents by **actual** HDS/MNDS status.

In [ ]:
COLORS={0:'#1261A0',1:'#E04A24'}
GROUP_NAMES={0:'Actual MNDS',1:'Actual HDS'}
actual=y_test.to_numpy(dtype=int)
ice_cache={feat:sweep(X_test,feat) for feat in plot_features}
ice_rows=[];category_count_rows=[];overlap_rows=[];boundary_rows=[];transition_rows=[]

from matplotlib.lines import Line2D


def add_ice_rug(ax, feat, group=None, *, data=None, outcome=None, kind=None, grid=None):
    if not SHOW_RUG_MARKS:
        return
    data = X_test if data is None else data
    outcome = actual if outcome is None else np.asarray(outcome,dtype=int)
    kind = feature_kind[feat] if kind is None else kind
    grid = grids[feat] if grid is None else np.asarray(grid,dtype=float)
    observed = data[feat].to_numpy(dtype=float)
    if group is not None:
        observed=observed[outcome == group]
    observed=observed[np.isfinite(observed)]
    if not len(observed):
        return
    if kind in NUMERIC_KINDS:
        # Numeric sentinel values denote skip/missing, not low numerical observations.
        if feat in NUMERIC_SENTINEL_FEATURES or feat in CATEGORICAL_FEATURES:
            observed=observed[~np.isclose(observed,SENTINEL)]
        x=observed
    else:
        nearest=np.argmin(abs(observed[:,None]-grid[None,:]),axis=1)
        valid=np.isclose(observed,grid[nearest])
        x=nearest[valid].astype(float)
        if not len(x):
            return
        rng=np.random.default_rng(RUG_SEED+sum(ord(ch) for ch in feat)+(0 if group is None else group))
        x+=rng.uniform(-0.17,0.17,size=len(x))
    if not len(x):
        return
    color='#777777' if group is None else COLORS[group]
    ax.vlines(x, RUG_HEIGHT[0],RUG_HEIGHT[1],transform=ax.get_xaxis_transform(),
              clip_on=True,color=color,alpha=.36,lw=.67,zorder=12)
    ax.tick_params(axis='x',pad=3)

def group_matrix(feat,group):
    return ice_cache[feat][actual==group]

def _plot_group(ax,feat,mat,group,centered=False):
    kind=feature_kind[feat];x=xvals(feat);color=COLORS[group]
    if not len(mat): return
    if centered:
        if kind not in NUMERIC_KINDS: raise ValueError('c-ICE only for numerical predictors.')
        mat=mat-mat[:,[0]]
    if kind in NUMERIC_KINDS:
        for curve in mat: ax.plot(x,curve,color=color,alpha=.18,lw=.8)
        ax.plot(x,np.median(mat,axis=0),color=color,lw=2.5)
    elif kind in ORDINAL_KINDS:
        valid_mask=~np.isclose(grids[feat],SENTINEL) if (
            feat in CATEGORICAL_FEATURES or feat in NUMERIC_SENTINEL_FEATURES) else np.ones(len(x),dtype=bool)
        if valid_mask.sum()>1:
            for curve in mat: ax.plot(np.asarray(x)[valid_mask],curve[valid_mask],
                                      color=color,alpha=.15,lw=.7)
            ax.plot(np.asarray(x)[valid_mask],np.median(mat[:,valid_mask],axis=0),
                    color=color,lw=2.4)
        ax.scatter(np.tile(x,len(mat)),mat.ravel(),color=color,s=9,alpha=.08,rasterized=True)
        ax.scatter(x,np.median(mat,axis=0),color=color,marker='D',s=32,
                   edgecolors='white',linewidth=.4,zorder=5)
    else:
        rng=np.random.default_rng(NB6_SEED+group)
        xx=np.broadcast_to(np.asarray(x)[None,:],mat.shape)+rng.uniform(-.08,.08,mat.shape)
        ax.scatter(xx.ravel(),mat.ravel(),color=color,alpha=.16,s=12,rasterized=True)
        ax.scatter(x,np.median(mat,axis=0),color=color,marker='D',s=38,
                   edgecolors='white',linewidth=.4,zorder=5)

def draw_ice(ax,feat,group,centered=False,show_count_labels=True):
    mat=group_matrix(feat,group);x=xvals(feat)
    if not len(mat): return
    _plot_group(ax,feat,mat,group,centered=centered)
    if feature_kind[feat] not in NUMERIC_KINDS:
        format_axis(ax,feat,group if show_count_labels else None)
    if centered:
        ax.axhline(0,color='#303030',lw=1,ls='--')
        ax.set_ylabel('Change in P(HDS) from anchor')
    else:
        ax.axhline(THRESHOLD,color='#303030',lw=1,ls='--')
        ax.set_ylim(-.04,1.04);ax.set_ylabel('P(HDS)')
    ax.set_xlabel(feat)
    ax.set_title(f'{GROUP_NAMES[group]} (n={int((actual==group).sum())})',fontsize=10,color=COLORS[group])
    ax.legend(handles=[Line2D([0],[0],color=COLORS[group],lw=1.2,alpha=.8,label=GROUP_NAMES[group]),
                       Line2D([0],[0],color=COLORS[group],lw=2.5,label='Group median'),
                       Line2D([0],[0],color='#303030',ls='--',lw=1,
                              label='Centered reference (0)' if centered else f'Threshold {THRESHOLD:.2f}')],
              fontsize=7,loc='best',framealpha=.88)
    ax.grid(alpha=.17)
    add_ice_rug(ax,feat,group=group)

def draw_category_counts(ax,feat,group):
    vals=X_test.loc[actual==group,feat].to_numpy(dtype=float)
    g=grids[feat];x=np.arange(len(g))
    counts=np.array([np.isclose(vals,v).sum() for v in g],dtype=int)
    ax.bar(x,counts,color=COLORS[group],alpha=.72,width=.68)
    ax.set_ylabel('Observed n',fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([textwrap.fill(level_label(feat,v),18) for v in g],
                       rotation=30,ha='right',fontsize=8)
    ax.grid(axis='y',alpha=.15)
    for xx,nn in zip(x,counts): ax.text(xx,nn,str(int(nn)),ha='center',va='bottom',fontsize=7)

def plot_feature(feat,centered=False,save=True):
    kind=feature_kind[feat]
    categorical=kind not in NUMERIC_KINDS
    if centered and categorical:return
    if categorical:
        fig,axes=plt.subplots(2,2,figsize=(15.8,8.5),gridspec_kw={'height_ratios':[3.2,1.1]})
        for col,group in enumerate((0,1)):
            draw_ice(axes[0,col],feat,group,centered=False,show_count_labels=False)
            draw_category_counts(axes[1,col],feat,group)
    else:
        fig,axes=plt.subplots(1,2,figsize=(15.4,4.9),sharey=True)
        for col,group in enumerate((0,1)):
            draw_ice(axes[col],feat,group,centered=centered)
    fig.suptitle(f'{"Centered ICE" if centered else "ICE"} | WINNING MODEL: {SELECTED_MODEL} | {feat} ({kind})',fontsize=13)
    fig.tight_layout(rect=(0,0,1,.963))
    if save:
        prefix='cICE' if centered else 'ICE'
        fig.savefig(NB6_DIR/f'{prefix}_{safe_name(feat)}.png',dpi=140,bbox_inches='tight')
    plt.show();plt.close(fig)

for feat in plot_features:
    mat=ice_cache[feat];kind=feature_kind[feat];x=grids[feat]
    swing=np.ptp(mat,axis=1)
    crossing=(mat.min(axis=1)<THRESHOLD)&(mat.max(axis=1)>=THRESHOLD)
    hds=group_matrix(feat,1);mnds=group_matrix(feat,0)
    # Overlap diagnostic: whether the two groups' 10%-90% prediction intervals
    # overlap at the same grid point. This measures overlap; it cannot rule it out.
    if len(hds) and len(mnds):
        h_low,h_high=np.percentile(hds,[10,90],axis=0)
        m_low,m_high=np.percentile(mnds,[10,90],axis=0)
        overlap=(h_low<=m_high)&(m_low<=h_high)
    else:overlap=np.zeros(len(x),dtype=bool)
    overlap_rows.append({'feature':feat,'kind':kind,
        'n_grid_points':len(x),'n_overlapping_10_90_intervals':int(overlap.sum()),
        'overlap_fraction':float(overlap.mean()) if len(x) else np.nan,
        'definition':'HDS/MNDS group 10th-90th percentile intervals overlap at same grid value'})
    ice_rows.append({'feature':feat,'kind':kind,'n_obs':len(X_test),
        'n_actual_HDS':int((actual==1).sum()),'n_actual_MNDS':int((actual==0).sum()),
        'mean_individual_range':float(swing.mean()),
        'median_individual_range':float(np.median(swing)),
        'n_curves_crossing_threshold':int(crossing.sum()),
        'n_actual_HDS_curves_crossing':int(crossing[actual==1].sum()),
        'n_actual_MNDS_curves_crossing':int(crossing[actual==0].sum())})
    for group in (0,1):
        for v in x if kind not in NUMERIC_KINDS else []:
            count=int(np.isclose(X_test.loc[actual==group,feat].to_numpy(dtype=float),v).sum())
            category_count_rows.append({'feature':feat,'kind':kind,
                'observed_group':GROUP_NAMES[group],'code':float(v),
                'level':level_label(feat,v),'n':count,
                'group_n':int((actual==group).sum())})
    # Boundary cases: at least one grid value changes the original predicted class.
    # They are hypothetical one-feature changes, NOT real observed transitions.
    for i in np.flatnonzero(crossing):
        changed_positions=np.flatnonzero((mat[i]>=THRESHOLD).astype(int)!=int(pred[i]))
        boundary_rows.append({'feature':feat,'test_index':X_test.index[i],
            'actual_HDS':int(actual[i]),'original_pred_HDS':int(pred[i]),
            'original_P_HDS':float(proba[i]),'original_feature_value':float(X_test.iloc[i][feat]),
            'n_grid_values_changing_prediction':int(len(changed_positions)),
            'first_changing_grid_value':float(x[changed_positions[0]]) if len(changed_positions) else np.nan,
            'first_changing_grid_label':level_label(feat,x[changed_positions[0]]) if len(changed_positions) else ''})
    if kind in NUMERIC_KINDS and len(x)>=3:
        d1=np.diff(mat,axis=1)
        d2=np.diff(mat,n=2,axis=1)
        for j in range(d1.shape[1]):
            transition_rows.append({'feature':feat,'from_value':float(x[j]),'to_value':float(x[j+1]),
                'mean_delta_P_HDS':float(d1[:,j].mean()),
                'mean_absolute_delta_P_HDS':float(np.abs(d1[:,j]).mean()),
                'n_curves_with_nonzero_delta':int((np.abs(d1[:,j])>1e-8).sum()),
                'mean_absolute_second_difference_at_next_grid':float(np.abs(d2[:,j]).mean())
                    if j<d2.shape[1] else np.nan})

ice_table_df=pd.DataFrame(ice_rows).sort_values('mean_individual_range',ascending=False)
ice_table_df.to_csv(NB6_DIR/'NB6_ICE_summary_table.csv',index=False)
pd.DataFrame(category_count_rows,columns=['feature','kind','observed_group','code','level','n','group_n']).to_csv(
    NB6_DIR/'NB6_ICE_category_counts.csv',index=False)
pd.DataFrame(overlap_rows).to_csv(NB6_DIR/'NB6_ICE_group_overlap_diagnostic.csv',index=False)
pd.DataFrame(boundary_rows,columns=['feature','test_index','actual_HDS','original_pred_HDS',
    'original_P_HDS','original_feature_value','n_grid_values_changing_prediction',
    'first_changing_grid_value','first_changing_grid_label']).to_csv(
    NB6_DIR/'NB6_ICE_boundary_cases.csv',index=False)
pd.DataFrame(transition_rows,columns=['feature','from_value','to_value','mean_delta_P_HDS',
    'mean_absolute_delta_P_HDS','n_curves_with_nonzero_delta',
    'mean_absolute_second_difference_at_next_grid']).to_csv(
    NB6_DIR/'NB6_ICE_finite_difference_diagnostics.csv',index=False)
print('ICE SUMMARY (ALL FEATURES)')
print(ice_table_df.round(4).to_string(index=False))
print('Saved group-overlap, threshold-crossing, category-count and finite-difference diagnostics.')

### **1A. Combined regular ICE (HDS and MNDS together)**

All test respondents are drawn in the **same axes**. Rug marks inside the lower edge of each axis show observed predictor values from all test respondents. For categorical predictors, horizontal rug jitter is for visibility only, while category labels remain discrete.

In [ ]:
def draw_pooled_ice(ax, feat, centered=False):
    kind=feature_kind[feat]
    if centered and kind not in NUMERIC_KINDS:
        raise ValueError('Centered ICE uses numeric predictors in this notebook.')
    for group in (0,1):
        mat=group_matrix(feat,group)
        _plot_group(ax,feat,mat,group,centered=centered)
    if kind not in NUMERIC_KINDS:
        format_axis(ax,feat)
    ax.axhline(0 if centered else THRESHOLD,color='#303030',lw=1,ls='--')
    if centered:
        ax.set_ylabel('Change in P(HDS) from anchor')
    else:
        ax.set_ylim(-.04,1.04)
        ax.set_ylabel('P(HDS)')
    ax.set_xlabel(feat)
    ax.grid(alpha=.17)
    add_ice_rug(ax,feat,group=None)
    ax.legend(handles=[Line2D([0],[0],color=COLORS[g],lw=2,
                              label=f'{GROUP_NAMES[g]} (n={sum(actual==g)})') for g in (0,1)] +
                      [Line2D([0],[0],color='#303030',ls='--',lw=1,
                              label='Centered reference (0)' if centered else f'Threshold {THRESHOLD:.2f}')],
              fontsize=7,framealpha=.88,loc='best')

# Plot calls are placed in separate cells below (one feature per cell).


### **1B. Separated regular ICE (actual MNDS versus actual HDS)**

The identical regular ICE curves appear in two panels to make overlapping profiles easier to read. Each panel's rug represents only test respondents with that **actual** HDS/MNDS status. Per-feature categorical plots retain the observed response-count panels.

In [ ]:
if SHOW_INDIVIDUAL_PLOTS:
    for feat in plot_features:
        plot_feature(feat,centered=False)

### **Individual regular ICE plots**

Each predictor has two separate outputs: pooled ICE (both actual groups in one axes) and separated ICE (MNDS/HDS panels). Each output is generated in its own cell.

#### Pooled regular ICE — predictor 1

In [ ]:
feat = plot_features[0]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 1

In [ ]:
feat = plot_features[0]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 2

In [ ]:
feat = plot_features[1]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 2

In [ ]:
feat = plot_features[1]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 3

In [ ]:
feat = plot_features[2]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 3

In [ ]:
feat = plot_features[2]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 4

In [ ]:
feat = plot_features[3]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 4

In [ ]:
feat = plot_features[3]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 5

In [ ]:
feat = plot_features[4]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 5

In [ ]:
feat = plot_features[4]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 6

In [ ]:
feat = plot_features[5]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 6

In [ ]:
feat = plot_features[5]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 7

In [ ]:
feat = plot_features[6]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 7

In [ ]:
feat = plot_features[6]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 8

In [ ]:
feat = plot_features[7]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 8

In [ ]:
feat = plot_features[7]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 9

In [ ]:
feat = plot_features[8]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 9

In [ ]:
feat = plot_features[8]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 10

In [ ]:
feat = plot_features[9]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 10

In [ ]:
feat = plot_features[9]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 11

In [ ]:
feat = plot_features[10]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 11

In [ ]:
feat = plot_features[10]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 12

In [ ]:
feat = plot_features[11]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 12

In [ ]:
feat = plot_features[11]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 13

In [ ]:
feat = plot_features[12]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 13

In [ ]:
feat = plot_features[12]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 14

In [ ]:
feat = plot_features[13]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 14

In [ ]:
feat = plot_features[13]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 15

In [ ]:
feat = plot_features[14]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 15

In [ ]:
feat = plot_features[14]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 16

In [ ]:
feat = plot_features[15]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 16

In [ ]:
feat = plot_features[15]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 17

In [ ]:
feat = plot_features[16]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 17

In [ ]:
feat = plot_features[16]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 18

In [ ]:
feat = plot_features[17]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 18

In [ ]:
feat = plot_features[17]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 19

In [ ]:
feat = plot_features[18]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 19

In [ ]:
feat = plot_features[18]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 20

In [ ]:
feat = plot_features[19]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 20

In [ ]:
feat = plot_features[19]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 21

In [ ]:
feat = plot_features[20]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 21

In [ ]:
feat = plot_features[20]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 22

In [ ]:
feat = plot_features[21]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 22

In [ ]:
feat = plot_features[21]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 23

In [ ]:
feat = plot_features[22]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 23

In [ ]:
feat = plot_features[22]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 24

In [ ]:
feat = plot_features[23]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 24

In [ ]:
feat = plot_features[23]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 25

In [ ]:
feat = plot_features[24]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 25

In [ ]:
feat = plot_features[24]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 26

In [ ]:
feat = plot_features[25]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 26

In [ ]:
feat = plot_features[25]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 27

In [ ]:
feat = plot_features[26]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 27

In [ ]:
feat = plot_features[26]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 28

In [ ]:
feat = plot_features[27]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 28

In [ ]:
feat = plot_features[27]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 29

In [ ]:
feat = plot_features[28]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 29

In [ ]:
feat = plot_features[28]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 30

In [ ]:
feat = plot_features[29]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 30

In [ ]:
feat = plot_features[29]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 31

In [ ]:
feat = plot_features[30]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 31

In [ ]:
feat = plot_features[30]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 32

In [ ]:
feat = plot_features[31]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 32

In [ ]:
feat = plot_features[31]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 33

In [ ]:
feat = plot_features[32]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 33

In [ ]:
feat = plot_features[32]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 34

In [ ]:
feat = plot_features[33]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 34

In [ ]:
feat = plot_features[33]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 35

In [ ]:
feat = plot_features[34]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 35

In [ ]:
feat = plot_features[34]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 36

In [ ]:
feat = plot_features[35]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 36

In [ ]:
feat = plot_features[35]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 37

In [ ]:
feat = plot_features[36]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 37

In [ ]:
feat = plot_features[36]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 38

In [ ]:
feat = plot_features[37]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 38

In [ ]:
feat = plot_features[37]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 39

In [ ]:
feat = plot_features[38]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 39

In [ ]:
feat = plot_features[38]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 40

In [ ]:
feat = plot_features[39]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 40

In [ ]:
feat = plot_features[39]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 41

In [ ]:
feat = plot_features[40]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 41

In [ ]:
feat = plot_features[40]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 42

In [ ]:
feat = plot_features[41]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 42

In [ ]:
feat = plot_features[41]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 43

In [ ]:
feat = plot_features[42]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 43

In [ ]:
feat = plot_features[42]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 44

In [ ]:
feat = plot_features[43]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 44

In [ ]:
feat = plot_features[43]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 45

In [ ]:
feat = plot_features[44]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 45

In [ ]:
feat = plot_features[44]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 46

In [ ]:
feat = plot_features[45]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 46

In [ ]:
feat = plot_features[45]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 47

In [ ]:
feat = plot_features[46]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 47

In [ ]:
feat = plot_features[46]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 48

In [ ]:
feat = plot_features[47]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 48

In [ ]:
feat = plot_features[47]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 49

In [ ]:
feat = plot_features[48]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 49

In [ ]:
feat = plot_features[48]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 50

In [ ]:
feat = plot_features[49]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 50

In [ ]:
feat = plot_features[49]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 51

In [ ]:
feat = plot_features[50]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 51

In [ ]:
feat = plot_features[50]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 52

In [ ]:
feat = plot_features[51]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 52

In [ ]:
feat = plot_features[51]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 53

In [ ]:
feat = plot_features[52]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 53

In [ ]:
feat = plot_features[52]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 54

In [ ]:
feat = plot_features[53]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 54

In [ ]:
feat = plot_features[53]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 55

In [ ]:
feat = plot_features[54]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 55

In [ ]:
feat = plot_features[54]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 56

In [ ]:
feat = plot_features[55]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 56

In [ ]:
feat = plot_features[55]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 57

In [ ]:
feat = plot_features[56]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 57

In [ ]:
feat = plot_features[56]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 58

In [ ]:
feat = plot_features[57]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 58

In [ ]:
feat = plot_features[57]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 59

In [ ]:
feat = plot_features[58]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 59

In [ ]:
feat = plot_features[58]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 60

In [ ]:
feat = plot_features[59]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 60

In [ ]:
feat = plot_features[59]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 61

In [ ]:
feat = plot_features[60]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 61

In [ ]:
feat = plot_features[60]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 62

In [ ]:
feat = plot_features[61]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 62

In [ ]:
feat = plot_features[61]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 63

In [ ]:
feat = plot_features[62]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 63

In [ ]:
feat = plot_features[62]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 64

In [ ]:
feat = plot_features[63]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 64

In [ ]:
feat = plot_features[63]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 65

In [ ]:
feat = plot_features[64]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 65

In [ ]:
feat = plot_features[64]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 66

In [ ]:
feat = plot_features[65]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 66

In [ ]:
feat = plot_features[65]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 67

In [ ]:
feat = plot_features[66]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 67

In [ ]:
feat = plot_features[66]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

#### Pooled regular ICE — predictor 68

In [ ]:
feat = plot_features[67]
print('Plotting pooled ICE:', feat)
fig, ax = plt.subplots(figsize=(8, 5))
draw_pooled_ice(ax, feat, centered=False)
ax.set_title(f'Pooled ICE | {feat} ({feature_kind[feat]})')
fig.tight_layout()
fig.savefig(NB6_DIR/f'ICE_pooled_{safe_name(feat)}.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

#### Separated regular ICE — predictor 68

In [ ]:
feat = plot_features[67]
print('Plotting separated ICE:', feat)
plot_feature(feat, centered=False)

### **1B2. Separated regular ICE: all predictors overview**

### **2. Centered ICE — change relative to a common reference**

Each respondent's curve is **centered at its prediction for the first value** of the predictor's training-derived numerical grid; the reference curve value is therefore 0. This helps compare modeled changes when absolute probabilities vary widely. Centered ICE is restricted here to numeric predictors; the categorical predictors are already handled as discrete comparisons in the regular ICE section. Rug marks continue to show **actual test observations**, not hypothetical grid points.

### **2A. Combined centered ICE (HDS and MNDS together)**

In [ ]:
centered_feats=[f for f in plot_features if feature_kind[f] in NUMERIC_KINDS and len(grids[f])>1]
print('Centered ICE numeric predictors:', len(centered_feats))

### **2B. Separated centered ICE (actual MNDS versus actual HDS)**

### **Individual centered ICE plots**

Centered ICE is limited to numeric predictors. Each cell generates one predictor figure.

#### Centered ICE — numeric predictor 1

In [ ]:
feat = centered_feats[0]
print('Plotting centered ICE:', feat)
plot_feature(feat, centered=True)

#### Centered ICE — numeric predictor 2

In [ ]:
feat = centered_feats[1]
print('Plotting centered ICE:', feat)
plot_feature(feat, centered=True)

#### Centered ICE — numeric predictor 3

In [ ]:
feat = centered_feats[2]
print('Plotting centered ICE:', feat)
plot_feature(feat, centered=True)

#### Centered ICE — numeric predictor 4

In [ ]:
feat = centered_feats[3]
print('Plotting centered ICE:', feat)
plot_feature(feat, centered=True)

#### Centered ICE — numeric predictor 5

In [ ]:
feat = centered_feats[4]
print('Plotting centered ICE:', feat)
plot_feature(feat, centered=True)

#### Centered ICE — numeric predictor 6

In [ ]:
feat = centered_feats[5]
print('Plotting centered ICE:', feat)
plot_feature(feat, centered=True)

#### Centered ICE — numeric predictor 7

In [ ]:
feat = centered_feats[6]
print('Plotting centered ICE:', feat)
plot_feature(feat, centered=True)

#### Centered ICE — numeric predictor 8

In [ ]:
feat = centered_feats[7]
print('Plotting centered ICE:', feat)
plot_feature(feat, centered=True)

#### Centered ICE — numeric predictor 9

In [ ]:
feat = centered_feats[8]
print('Plotting centered ICE:', feat)
plot_feature(feat, centered=True)

#### Centered ICE — numeric predictor 10

In [ ]:
feat = centered_feats[9]
print('Plotting centered ICE:', feat)
plot_feature(feat, centered=True)

The **rug marks are distribution cues**, not sample weights, prevalence estimates, or counts of predicted HDS cases. For numeric predictors, each tick is an observed test value; for categorical predictors, tiny horizontal offsets prevent complete overlap at identical labels. This is a visualization of conditional model behavior, not a causal effect.

### **ICE summary and reproducibility**

In [ ]:
summary=ice_table_df.merge(
    pd.DataFrame(overlap_rows)[['feature','n_overlapping_10_90_intervals','overlap_fraction']],
    on='feature',how='left')
summary.to_csv(NB6_DIR/'NB6_ICE_group_summary.csv',index=False)
print('ICE GROUP SUMMARY')
print(summary.round(4).to_string(index=False))
print('Saved ICE-only outputs to',NB6_DIR)

# Reproducibility: saved model and held-out dataset are not modified.
def file_sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()

inputs={'model':OUT_DIR/f'NB4_fitted_{SELECTED_MODEL}.joblib',
        'train':OUT_DIR/TRAIN_FILE,'test':OUT_DIR/TEST_FILE,
        'selection':OUT_DIR/'NB5_model_selection.json',
        'ranking':OUT_DIR/'NB5_training_only_ranking.csv',
        'predictions':OUT_DIR/'NB5_combined_predictions.csv',
        'SPSS_level_map':LEVEL_FILE}
manifest={'notebook':'NB6', 'revision':'consultation ICE; regular/centered pooled+separated, test-observation rug marks, RF comparison; type audit',
    'selected_model':SELECTED_MODEL,'decision_threshold':THRESHOLD,
    'seed_for_visual_jitter':NB6_SEED,'train_n':len(X_train),'test_n':len(X_test),
    'features':FEATURES,'grid_points':GRID_POINTS,'predictor_type_audit':'NB6_predictor_type_audit.csv',
    'library_versions':{'python':sys.version.split()[0],
        'numpy':np.__version__,'pandas':pd.__version__,
        'sklearn':sklearn.__version__,'matplotlib':matplotlib.__version__},
    'input_sha256':{name:file_sha256(path) for name,path in inputs.items()},
    'selection_method':'Unchanged NB5. No retuning, refitting, or reselection in NB6.',
    'plots':'Regular ICE pooled + separated by actual outcome; numeric centered ICE pooled + separated; held-out-test rug marks for every ICE panel.',
    'derivative_note':'For stepwise RF predictions, discrete finite differences are diagnostic, not smooth derivatives.'}

(NB6_DIR/'NB6_reproducibility_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Saved reproducibility manifest:',NB6_DIR/'NB6_reproducibility_manifest.json')

### **Baseline versus tuned Random Forest comparison**

The existing tuned-versus-baseline RF comparison is retained without CP plots. It is supplementary and does not change the NB5 model selection.

In [ ]:
rf_train=pd.read_csv(OUT_DIR/'NB3_RandomForest_train.csv',index_col='orig_index')
rf_test=pd.read_csv(OUT_DIR/'NB3_RandomForest_test.csv',index_col='orig_index')
RF_FEATURES=[c for c in rf_train.columns if c!='depressed_flag']
rf_xtr=rf_train[RF_FEATURES].astype(float)
rf_xt=rf_test[RF_FEATURES].astype(float)
rf_y=rf_test['depressed_flag'].astype(int).to_numpy()
rf_tuned=joblib.load(OUT_DIR/'NB4_fitted_RF_tuned.joblib')
rf_unrestricted=joblib.load(OUT_DIR/'NB4_fitted_RF_baseline.joblib')
rf_tuned_thr=float(ranking.loc['RF_tuned','threshold'])
rf_unrestricted_thr=float(ranking.loc['RF_baseline','threshold'])

comparison_source=nb5[nb5['split'].eq('test')].set_index('original_idx').loc[rf_xt.index]
for name,est in [('RF_tuned',rf_tuned),('RF_baseline',rf_unrestricted)]:
    assert np.allclose(est.predict_proba(rf_xt)[:,1],comparison_source[name+'_proba'].to_numpy(),atol=1e-8)

comparison_dir=OUT_DIR/'NB6_RF_restricted_vs_unrestricted'
comparison_dir.mkdir(exist_ok=True,parents=True)
RF_COMPARISON_ALL=True
rf_comparison_features=RF_FEATURES if RF_COMPARISON_ALL else RF_FEATURES[:12]
rf_comp_rows=[]
rf_feature_grids={};rf_feature_kind={}
for feat in RF_FEATURES:
    if feat in grids:
        rf_feature_grids[feat]=grids[feat]
        rf_feature_kind[feat]=feature_kind[feat]
        continue
    observed=rf_xtr[feat].dropna().to_numpy(dtype=float)
    kind,grid,_,_=choose_grid_kind(feat,observed)
    rf_feature_grids[feat]=np.asarray(grid,dtype=float)
    rf_feature_kind[feat]=kind

def rf_sweep(est,rows,feat,grid):
    n=len(rows);k=len(grid)
    df=pd.DataFrame(np.repeat(rows[RF_FEATURES].to_numpy(),k,axis=0),columns=RF_FEATURES)
    df[feat]=np.tile(grid,n)
    return est.predict_proba(df)[:,1].reshape(n,k)

def draw_rf_comp(ax,mat,feat,group,model_name,thr):
    kind=rf_feature_kind[feat]
    grid=rf_feature_grids[feat]
    x=grid if kind in NUMERIC_KINDS else np.arange(len(grid))
    color=COLORS[group]
    sub=mat[rf_y==group]
    if kind in NUMERIC_KINDS:
        for v in sub:ax.plot(x,v,color=color,alpha=.2,lw=.8)
        ax.plot(x,np.median(sub,axis=0),color=color,lw=2.4)
    elif kind in ORDINAL_KINDS:
        valid_mask=~np.isclose(grid,SENTINEL) if (feat in CATEGORICAL_FEATURES or
            feat in NUMERIC_SENTINEL_FEATURES) else np.ones(len(grid),dtype=bool)
        if valid_mask.sum()>1:
            for v in sub:ax.plot(np.asarray(x)[valid_mask],v[valid_mask],color=color,alpha=.15,lw=.8)
            ax.plot(np.asarray(x)[valid_mask],np.median(sub[:,valid_mask],axis=0),color=color,lw=2.4)
        ax.scatter(np.tile(x,len(sub)),sub.ravel(),color=color,alpha=.08,s=9)
        ax.scatter(x,np.median(sub,axis=0),color=color,marker='D',s=35)
        ax.set_xticks(x)
        ax.set_xticklabels([textwrap.fill(level_label(feat,v),18) for v in grid],rotation=30,ha='right',fontsize=8)
    else:
        for v in sub:ax.scatter(x,v,color=color,alpha=.12,s=12)
        ax.scatter(x,np.median(sub,axis=0),color=color,marker='D',s=35)
        ax.set_xticks(x)
        ax.set_xticklabels([textwrap.fill(level_label(feat,v),18) for v in grid],rotation=30,ha='right',fontsize=8)
    ax.axhline(thr,color='#303030',lw=1,ls='--')
    ax.set_ylim(-.04,1.04);ax.set_xlabel(feat);ax.set_ylabel('P(HDS)')
    ax.grid(alpha=.15)
    ax.set_title(f'{model_name} | {GROUP_NAMES[group]} (n={sum(rf_y==group)})',fontsize=10)
    add_ice_rug(ax,feat,group=group,data=rf_xt,outcome=rf_y,kind=kind,grid=grid)
    ax.legend(handles=[Line2D([0],[0],color=color,lw=2,label=GROUP_NAMES[group]),
                       Line2D([0],[0],color='#303030',ls='--',lw=1,
                              label=f'Threshold {thr:.2f}')],fontsize=7)
    
def plot_rf_comparison_feature(feat):
    """Plot one predictor's RF_tuned versus RF_baseline ICE comparison."""
    grid=rf_feature_grids[feat]
    vals_tuned=rf_sweep(rf_tuned,rf_xt,feat,grid)
    vals_unres=rf_sweep(rf_unrestricted,rf_xt,feat,grid)
    fig,axes=plt.subplots(2,2,figsize=(14,9),sharey=True)
    for col,(title,mat,thr,key) in enumerate([
        ('RF_tuned (WINNING MODEL)',vals_tuned,rf_tuned_thr,'tuned'),
        ('RF_baseline',vals_unres,rf_unrestricted_thr,'unrestricted')]):
        for row,group in enumerate((0,1)):
            draw_rf_comp(axes[row,col],mat,feat,group,title,thr)
    fig.suptitle(f'{feat}: tuned vs unrestricted RF | ICE only',fontsize=13)
    fig.tight_layout(rect=(0,0,1,.97))
    fig.savefig(comparison_dir/f'comparison_{safe_name(feat)}.png',dpi=120,bbox_inches='tight')
    plt.show();plt.close(fig)
    result={'feature':feat,'kind':rf_feature_kind[feat],
        'tuned_ICE_mean_range':float(np.ptp(vals_tuned,axis=1).mean()),
        'unrestricted_ICE_mean_range':float(np.ptp(vals_unres,axis=1).mean()),
        'tuned_ICE_threshold_crossings':int(((vals_tuned.min(axis=1)<rf_tuned_thr)&(vals_tuned.max(axis=1)>=rf_tuned_thr)).sum()),
        'unrestricted_ICE_threshold_crossings':int(((vals_unres.min(axis=1)<rf_unrestricted_thr)&(vals_unres.max(axis=1)>=rf_unrestricted_thr)).sum())}
    rf_comp_rows.append(result)
    return result

### **Individual baseline-versus-tuned RF ICE comparisons**

Each cell generates one predictor comparison.

#### RF comparison — predictor 1

In [ ]:
feat = rf_comparison_features[0]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 2

In [ ]:
feat = rf_comparison_features[1]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 3

In [ ]:
feat = rf_comparison_features[2]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 4

In [ ]:
feat = rf_comparison_features[3]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 5

In [ ]:
feat = rf_comparison_features[4]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 6

In [ ]:
feat = rf_comparison_features[5]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 7

In [ ]:
feat = rf_comparison_features[6]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 8

In [ ]:
feat = rf_comparison_features[7]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 9

In [ ]:
feat = rf_comparison_features[8]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 10

In [ ]:
feat = rf_comparison_features[9]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 11

In [ ]:
feat = rf_comparison_features[10]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 12

In [ ]:
feat = rf_comparison_features[11]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 13

In [ ]:
feat = rf_comparison_features[12]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 14

In [ ]:
feat = rf_comparison_features[13]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 15

In [ ]:
feat = rf_comparison_features[14]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 16

In [ ]:
feat = rf_comparison_features[15]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 17

In [ ]:
feat = rf_comparison_features[16]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 18

In [ ]:
feat = rf_comparison_features[17]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 19

In [ ]:
feat = rf_comparison_features[18]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 20

In [ ]:
feat = rf_comparison_features[19]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 21

In [ ]:
feat = rf_comparison_features[20]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 22

In [ ]:
feat = rf_comparison_features[21]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 23

In [ ]:
feat = rf_comparison_features[22]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 24

In [ ]:
feat = rf_comparison_features[23]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 25

In [ ]:
feat = rf_comparison_features[24]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 26

In [ ]:
feat = rf_comparison_features[25]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 27

In [ ]:
feat = rf_comparison_features[26]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 28

In [ ]:
feat = rf_comparison_features[27]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 29

In [ ]:
feat = rf_comparison_features[28]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 30

In [ ]:
feat = rf_comparison_features[29]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 31

In [ ]:
feat = rf_comparison_features[30]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 32

In [ ]:
feat = rf_comparison_features[31]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 33

In [ ]:
feat = rf_comparison_features[32]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 34

In [ ]:
feat = rf_comparison_features[33]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 35

In [ ]:
feat = rf_comparison_features[34]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 36

In [ ]:
feat = rf_comparison_features[35]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 37

In [ ]:
feat = rf_comparison_features[36]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 38

In [ ]:
feat = rf_comparison_features[37]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 39

In [ ]:
feat = rf_comparison_features[38]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 40

In [ ]:
feat = rf_comparison_features[39]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 41

In [ ]:
feat = rf_comparison_features[40]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 42

In [ ]:
feat = rf_comparison_features[41]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 43

In [ ]:
feat = rf_comparison_features[42]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 44

In [ ]:
feat = rf_comparison_features[43]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 45

In [ ]:
feat = rf_comparison_features[44]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 46

In [ ]:
feat = rf_comparison_features[45]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 47

In [ ]:
feat = rf_comparison_features[46]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 48

In [ ]:
feat = rf_comparison_features[47]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 49

In [ ]:
feat = rf_comparison_features[48]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 50

In [ ]:
feat = rf_comparison_features[49]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 51

In [ ]:
feat = rf_comparison_features[50]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 52

In [ ]:
feat = rf_comparison_features[51]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 53

In [ ]:
feat = rf_comparison_features[52]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 54

In [ ]:
feat = rf_comparison_features[53]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 55

In [ ]:
feat = rf_comparison_features[54]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 56

In [ ]:
feat = rf_comparison_features[55]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 57

In [ ]:
feat = rf_comparison_features[56]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 58

In [ ]:
feat = rf_comparison_features[57]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 59

In [ ]:
feat = rf_comparison_features[58]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 60

In [ ]:
feat = rf_comparison_features[59]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 61

In [ ]:
feat = rf_comparison_features[60]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 62

In [ ]:
feat = rf_comparison_features[61]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 63

In [ ]:
feat = rf_comparison_features[62]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 64

In [ ]:
feat = rf_comparison_features[63]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 65

In [ ]:
feat = rf_comparison_features[64]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 66

In [ ]:
feat = rf_comparison_features[65]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 67

In [ ]:
feat = rf_comparison_features[66]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

#### RF comparison — predictor 68

In [ ]:
feat = rf_comparison_features[67]
print('Comparing RF models for:', feat)
plot_rf_comparison_feature(feat)

### **Save RF comparison summary**

In [ ]:
rf_comparison_df=pd.DataFrame(rf_comp_rows)
rf_comparison_df['difference_ICE_range_tuned_minus_unrestricted']=(
    rf_comparison_df['tuned_ICE_mean_range']-rf_comparison_df['unrestricted_ICE_mean_range'])
rf_comparison_df.to_csv(comparison_dir/'NB6_tuned_vs_unrestricted_all_features.csv',index=False)
print('Tuned vs unrestricted RF comparison (ICE only):')
print(rf_comparison_df.round(4).to_string(index=False))
print('Saved to:',comparison_dir)

### **Overall interpretation**

This section will be completed after the ICE code and resulting figures are validated.